In [25]:
# ==========================================
# Project Name: GroupDNA - WhatsApp Chat Decoder
# NAME:Ayush Dayal
# AI-Assisted: Gemini helped structure the logic strictly to Python fundamentals and NumPy.


import numpy as np
from datetime import datetime, timedelta
import string

# Define the file path for the exported WhatsApp chat
FILE_PATH = 'hostel_bois.txt'

# ==========================================
# Feature 1: The Chat Parser
# ==========================================
def parse_chat(file_path):
    """Reads the text file, extracts messages, and handles edge cases like media and deleted messages."""
    parsed_messages = []

    # Dictionary to keep track of skipped messages (Edge Cases)
    stats = {
        'system': 0,
        'media': 0,
        'deleted': 0,
        'media_per_person': {},
        'deleted_per_person': {}
    }

    # Open the file safely using UTF-8 encoding to handle emojis/special characters
    with open(file_path, 'r', encoding='utf-8') as f:
        lines = f.readlines()

    for line in lines:
        line = line.strip() # Remove leading/trailing whitespace
        if not line:
            continue # Skip empty lines

        # WhatsApp format is usually: DD/MM/YY, HH:MM - Sender: Message
        # We split by ' - ' to separate the timestamp from the rest of the message
        parts = line.split(' - ', 1)

        # If there's no ' - ', it means this line is a continuation of the previous message
        if len(parts) != 2:
            if parsed_messages:
                # Append this line to the text of the last saved message
                parsed_messages[-1]['text'] += " " + line
            continue

        timestamp_str, rest = parts

        # Split the remaining part by ': ' to separate the sender's name from their text
        msg_parts = rest.split(': ', 1)

        # If there's no colon, it's a WhatsApp system message (e.g., "Priya added you")
        if len(msg_parts) != 2:
            stats['system'] += 1
            continue

        sender, text = msg_parts

        # Edge Case 1: Media omitted (photos, videos, stickers)
        if text == '<Media omitted>':
            stats['media'] += 1
            # Track who sends the most media
            stats['media_per_person'][sender] = stats['media_per_person'].get(sender, 0) + 1
            # We append it with type 'media' so we can exclude it from word counts later
            parsed_messages.append({'timestamp': timestamp_str, 'sender': sender, 'text': text, 'type': 'media'})
            continue

        # Edge Case 2: Deleted messages
        if text == 'This message was deleted':
            stats['deleted'] += 1
            stats['deleted_per_person'][sender] = stats['deleted_per_person'].get(sender, 0) + 1
            parsed_messages.append({'timestamp': timestamp_str, 'sender': sender, 'text': text, 'type': 'deleted'})
            continue

        # If it passes all checks, it's a valid text message
        parsed_messages.append({'timestamp': timestamp_str, 'sender': sender, 'text': text, 'type': 'text'})

    return parsed_messages, stats

# Run the parser and store results
messages, edge_stats = parse_chat(FILE_PATH)
real_msgs_count = len(messages)
# Use a Set to easily find unique participant names
unique_participants = set(msg['sender'] for msg in messages)

# ==========================================
# Feature 2: Group Overview
# ==========================================
def generate_overview(messages):
    """Calculates total days active and sorts participants by message count."""
    # Extract the date part (before the comma) from the very first and very last messages
    first_date_str = messages[0]['timestamp'].split(',')[0]
    last_date_str = messages[-1]['timestamp'].split(',')[0]

    # Convert string dates to datetime objects to calculate the difference in days
    d1 = datetime.strptime(first_date_str, '%d/%m/%y')
    d2 = datetime.strptime(last_date_str, '%d/%m/%y')
    total_days = (d2 - d1).days + 1

    # Count how many messages each person sent using a dictionary
    per_person_counts = {}
    for msg in messages:
        sender = msg['sender']
        per_person_counts[sender] = per_person_counts.get(sender, 0) + 1

    # Sort the dictionary by value (message count) in descending order
    sorted_counts = sorted(per_person_counts.items(), key=lambda item: item[1], reverse=True)

    return first_date_str, last_date_str, total_days, sorted_counts

first_date, last_date, total_days, sorted_participant_counts = generate_overview(messages)

# ==========================================
# Feature 3: Most Active Day and Hour
# ==========================================
def find_peak_activity(messages, total_days):
    """Finds the single day and single hour with the highest message volume."""
    day_counts = {}
    hour_counts = {}

    for msg in messages:
        ts = msg['timestamp']
        # Split "DD/MM/YY, HH:MM" into ["DD/MM/YY", "HH:MM"]
        date_part, time_part = ts.split(', ')
        # Extract just the hour "HH"
        hour_part = time_part.split(':')[0]

        day_counts[date_part] = day_counts.get(date_part, 0) + 1
        hour_counts[hour_part] = hour_counts.get(hour_part, 0) + 1

    # max() with a lambda function finds the key with the highest dictionary value
    busiest_day = max(day_counts.items(), key=lambda x: x[1])
    busiest_hour = max(hour_counts.items(), key=lambda x: x[1])

    # Calculate average messages per day during that specific peak hour
    avg_busiest_hour = busiest_hour[1] // total_days

    return busiest_day, busiest_hour, avg_busiest_hour

busiest_day_data, busiest_hour_data, avg_busiest_hour_msgs = find_peak_activity(messages, total_days)

# ==========================================
# Feature 4: NumPy Activity Heatmap
# ==========================================
def build_heatmap(messages, sorted_participants):
    """Uses NumPy to build a 2D matrix of Activity (Rows = People, Cols = 24 Hours)."""
    # Extract just the names in sorted order
    participants_list = [p[0] for p in sorted_participants]

    # Initialize a NumPy array filled with zeros (Shape: Number of People x 24 hours)
    heatmap_matrix = np.zeros((len(participants_list), 24), dtype=int)

    for msg in messages:
        sender = msg['sender']
        # Extract the hour as an integer to use as the column index
        hour = int(msg['timestamp'].split(', ')[1].split(':')[0])

        # Find the row index for this specific sender
        row_idx = participants_list.index(sender)
        # Increment the count in the matrix
        heatmap_matrix[row_idx, hour] += 1

    return heatmap_matrix, participants_list

activity_matrix, participant_names = build_heatmap(messages, sorted_participant_counts)

# ==========================================
# Feature 5: Top Words
# ==========================================
def get_top_words(messages):
    """Finds the most used words, excluding common English/Hindi stop words."""
    # A set of words to ignore (O(1) lookup time)
    stop_words = {'i', 'is', 'the', 'a', 'and', 'or', 'to', 'of', 'in', 'on', 'for',
                  'it', 'you', 'are', 'with', 'was', 'so', 'have', 'but', 'we', 'at',
                  'just', 'not', 'what', 'can', 'me', 'am', 'all', 'out', 'up', 'he',
                  'hai', 'hu', 'se', 'ki', 'ko', 'ye', 'woh', 'ke', 'ka', 'tu'}

    word_counts = {}

    for msg in messages:
        # Ignore media/deleted messages for word counts
        if msg['type'] != 'text':
            continue

        text = msg['text'].lower()

        # Remove punctuation using Python's built-in string translation
        translator = str.maketrans('', '', string.punctuation)
        text = text.translate(translator)

        words = text.split()
        for w in words:
            # Only count words that are not in the stop_words set
            if w not in stop_words:
                word_counts[w] = word_counts.get(w, 0) + 1

    # Sort words by their frequency in descending order and return the top words
    sorted_words = sorted(word_counts.items(), key=lambda item: item[1], reverse=True)
    return sorted_words


In [26]:
## Feature 2: Group Overview

# Build the group summary
def to_dt(ts):
    return datetime.strptime(ts, '%d/%m/%y, %H:%M')

if messages:
    dates = [to_dt(m['timestamp']).date() for m in messages]
    first_date, last_date = min(dates), max(dates)
    day_count = (last_date - first_date).days + 1
    per_person = {}
    for m in messages:
        per_person[m['sender']] = per_person.get(m['sender'], 0) + 1
    print('GROUP OVERVIEW')
    print(f'Period {first_date} to {last_date} ({day_count} days)')
    print(f'Total messages: {len(messages)}')
    print(f'Participants: {len(unique_participants)}')
    for name, count in sorted(per_person.items(), key=lambda x: x[1], reverse=True):
        print(f'{name:<10} {count}')

GROUP OVERVIEW
Period 2024-04-01 to 2024-05-30 (60 days)
Total messages: 3174
Participants: 6
Rahul      953
Priya      718
Neha       635
Aman       490
Karan      354
Vikas      24


In [27]:
## Feature 3: Busy Day and Hour

# Find the busiest day and hour
if messages:
    day_counts = {}
    hour_counts = {}
    for m in messages:
        d = str(to_dt(m['timestamp']).date())
        h = to_dt(m['timestamp']).hour
        day_counts[d] = day_counts.get(d, 0) + 1
        hour_counts[h] = hour_counts.get(h, 0) + 1
    busy_day = max(day_counts, key=day_counts.get)
    busy_hour = max(hour_counts, key=hour_counts.get)
    print(f'Busiest day: {busy_day} -> {day_counts[busy_day]} messages')
    print(f'Busiest hour: {busy_hour:02d}:00 -> {hour_counts[busy_hour]} messages')

Busiest day: 2024-05-04 -> 76 messages
Busiest hour: 18:00 -> 248 messages


In [28]:
## Feature 4: Heatmap

# Build a simple NumPy heatmap by person and hour
if messages and participant_names:
    idx = {p: i for i, p in enumerate(participant_names)}
    heat = np.zeros((len(participant_names), 24), dtype=int)
    for m in messages:
        heat[idx[m['sender']], to_dt(m['timestamp']).hour] += 1
    shades = [' ', '·', '░', '▓', '█']
    print('ACTIVITY HEATMAP')
    print('      ' + ' '.join(f'{h:02d}' for h in range(0, 24, 3)))
    for i, p in enumerate(participant_names):
        row = heat[i]
        mx = row.max() if row.max() else 1
        cells = []
        for v in row[::3]:
            level = int((v / mx) * 4) if mx else 0
            cells.append(shades[level])
        print(f'{p:<8} ' + ' '.join(cells))

ACTIVITY HEATMAP
      00 03 06 09 12 15 18 21
Rahul            ░ ░ █ ▓
Priya          █ ▓ · ░ ·
Neha           ▓ ░   █ ·
Aman     ░ ░            
Karan          · █ ░ ░ ·
Vikas          · ░ · ░ ·


In [29]:
## Feature 5: Top Words

# Count common words from real messages only
stop_words = {'i', 'is', 'the', 'a', 'and', 'or', 'to', 'of', 'in', 'on', 'for', 'me', 'you', 'it', 'this', 'that'}
freq = {}
for m in messages:
    if m['text'] in ('Media omitted', 'This message was deleted'):
        continue
    for w in m['text'].lower().split():
        word = w.strip('.,!?;:"\'()[]{}')
        if word and word not in stop_words:
            freq[word] = freq.get(word, 0) + 1
print('TOP WORDS')
for word, count in sorted(freq.items(), key=lambda x: x[1], reverse=True)[:10]:
    print(f'{word:<12} {count}')


TOP WORDS
was          385
how          321
guys         318
so           292
about        274
hai          268
am           260
today        257
at           257
my           223


In [30]:
## Feature 6: Response and Silence

# Estimate response gaps and silent streaks
if messages:
    gaps = {p: [] for p in participant_names}
    last_dt = None
    last_sender = None
    day_set = sorted({to_dt(m['timestamp']).date() for m in messages})
    per_day = {p: {} for p in participant_names}
    for m in messages:
        dt = to_dt(m['timestamp'])
        per_day[m['sender']][dt.date()] = per_day[m['sender']].get(dt.date(), 0) + 1
        if last_dt is not None and m['sender'] != last_sender:
            gaps[m['sender']].append((dt - last_dt).total_seconds() / 60)
        last_dt, last_sender = dt, m['sender']
    print('RESPONSE SPEED')
    for p in participant_names:
        avg = sum(gaps[p]) / len(gaps[p]) if gaps[p] else 0
        print(f'{p:<10} avg {avg:.1f} minutes')
    print('SILENT STREAKS')
    for p in participant_names:
        longest = 0
        current = 0
        for d in day_set:
            if d in per_day[p]:
                longest = max(longest, current)
                current = 0
            else:
                current += 1
        longest = max(longest, current)
        print(f'{p:<10} {longest} days')

RESPONSE SPEED
Rahul      avg 34.9 minutes
Priya      avg 42.0 minutes
Neha       avg 39.4 minutes
Aman       avg 55.4 minutes
Karan      avg 36.6 minutes
Vikas      avg 46.3 minutes
SILENT STREAKS
Rahul      0 days
Priya      0 days
Neha       0 days
Aman       0 days
Karan      0 days
Vikas      11 days


In [31]:
## Feature 7: Archetypes

# Give each person a simple scored archetype
def score_archetypes(person):
    texts = [m['text'] for m in messages if m['sender'] == person]
    count = len(texts) or 1
    words = sum(len(t.split()) for t in texts)
    night = sum(1 for m in messages if m['sender'] == person and (to_dt(m['timestamp']).hour >= 23 or to_dt(m['timestamp']).hour < 5))
    caps = sum(1 for t in texts if t.isupper() and len(t) > 2)
    q = sum(1 for t in texts if t.endswith('?'))
    scores = {
        'THE SPAMMER': count,
        'THE NIGHT OWL': night,
        'THE STORYTELLER': words / count,
        'THE DRAMA QUEEN': caps,
        'THE QUESTION MASTER': q,
        'THE GHOST': 60 - len({to_dt(m['timestamp']).date() for m in messages if m['sender'] == person}),
        'THE COMEDIAN': sum(1 for t in texts if any(x in t.lower() for x in ['lol', 'lmao', 'haha', 'rofl', 'lmfao'])),
        'THE GROUP MOM': sum(1 for t in texts if any(x in t.lower() for x in ['okay', 'safe', 'eat', 'sleep', 'please', 'reminder'])),
        'THE PUNCTUAL ONE': -sum(1 for m in messages if m['sender'] == person and to_dt(m['timestamp']).hour >= 23),
    }
    return max(scores, key=scores.get), scores

if messages:
    print('ARCHETYPES')
    for p in participant_names:
        archetype, scores = score_archetypes(p)
        print(f'{p:<10} {archetype}')

ARCHETYPES
Rahul      THE SPAMMER
Priya      THE SPAMMER
Neha       THE SPAMMER
Aman       THE SPAMMER
Karan      THE SPAMMER
Vikas      THE GHOST


In [32]:
## Feature 8: Final Report

# Print one clean report at the end
if messages:
    print('\n' + '=' * 40)
    print('GROUPDNA REPORT')
    print('=' * 40)
    print(f'Members: {len(unique_participants)}')
    print(f'Messages: {len(messages)}')
    print(f'System skipped: {edge_stats["system"]}, Media: {edge_stats["media"]}, Deleted: {edge_stats["deleted"]}')
    print('Done.')
else:
    print('No data loaded yet.')


GROUPDNA REPORT
Members: 6
Messages: 3174
System skipped: 4, Media: 32, Deleted: 15
Done.


In [33]:
## Reflection

## The hardest part is handling messy chat lines carefully without using forbidden libraries. I would improve the formatting next and make the archetype scoring more balanced.